# ADBE Revenue Normalization

This notebook teaches the transformation OwnerLens performs to turn raw SEC filings into a trustworthy annual revenue series.

```text
Raw SEC Company Facts
        ↓
Find candidate revenue concepts
        ↓
Inspect observations
        ↓
Understand:
  FY vs quarterly/YTD
  filing FY vs economic FY
  repeated comparatives
        ↓
Apply OwnerLens normalization
        ↓
Canonical 5-year revenue series
        ↓
Inspect provenance
```

> Set an SEC User-Agent that identifies your application and an administrative contact. Replace the sample contact below with a real one before making live requests.

## 1. Retrieve Adobe Company Facts

We use the Slice 1 `SecClient` to resolve ADBE and download its raw Company Facts. The payload is preserved unchanged: this is source truth, not yet a canonical fact.

In [1]:
import os

from owner_lens import SecClient, normalize_annual_revenue

# Replace the contact with your own before live use.
USER_AGENT = os.environ.get("OWNER_LENS_SEC_USER_AGENT", "OwnerLens admin@example.com")

with SecClient(USER_AGENT) as client:
    result = client.retrieve_company_facts("ADBE")

raw = result.raw_facts
print("ticker: ", result.identity.ticker)
print("company:", result.identity.company_name)
print("CIK:    ", result.identity.cik)
print("top-level keys:", list(raw.keys()))

ticker:  ADBE
company: ADOBE INC.
CIK:     0000796343
top-level keys: ['cik', 'entityName', 'facts']


## 2. Inspect available revenue concepts

The `us-gaap` taxonomy contains many revenue-named concepts. Most are not the top-line figure we want. OwnerLens picks one concept using a documented preference order, not by guessing.

In [2]:
from owner_lens.revenue import REVENUE_CONCEPT_PREFERENCE

us_gaap = raw["facts"]["us-gaap"]
revenue_like = sorted(k for k in us_gaap if "Revenue" in k)
print("revenue-like concepts:")
for name in revenue_like:
    print("  ", name)

print("\npreference order OwnerLens tries:")
for name in REVENUE_CONCEPT_PREFERENCE:
    present = "present" if name in us_gaap else "absent"
    print(f"  {name} ({present})")

revenue-like concepts:
   BusinessAcquisitionProFormaRevenue
   ContractWithCustomerLiabilityRevenueRecognized
   CostOfRevenue
   DeferredRevenue
   DeferredRevenueCurrent
   DeferredRevenueNoncurrent
   EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToIndividualForeignCountriesAmount
   EntityWideRevenueMajorCustomerPercentage
   IncreaseDecreaseInDeferredRevenue
   OperatingLeasesIncomeStatementSubleaseRevenue
   RevenueRemainingPerformanceObligation
   RevenueRemainingPerformanceObligationPercentage
   Revenues
   SalesRevenueGoodsNet
   SalesRevenueServicesNet
   SubscriptionRevenue

preference order OwnerLens tries:
  RevenueFromContractWithCustomerExcludingAssessedTax (absent)
  Revenues (present)
  SalesRevenueNet (absent)


## 3. Look at raw `Revenues` observations

Adobe reports top-line revenue under `Revenues` in USD. A single concept holds hundreds of observations: annual, quarterly, and year-to-date periods all mixed together.

In [3]:
usd = us_gaap["Revenues"]["units"]["USD"]
print("units available:", list(us_gaap["Revenues"]["units"].keys()))
print("total USD observations:", len(usd))
print("fact fields:", sorted(usd[0].keys()))


def duration_days(fact):
    from datetime import date

    return (date.fromisoformat(fact["end"]) - date.fromisoformat(fact["start"])).days


print("\nsample observations (period, days, fp, form, val):")
for fact in usd[:6]:
    print(
        f"  {fact['start']} -> {fact['end']}  {duration_days(fact):>3}d  "
        f"{fact['fp']:>3}  {fact['form']:<6}  {fact['val']:,}"
    )

units available: ['USD']
total USD observations: 313
fact fields: ['accn', 'end', 'filed', 'form', 'fp', 'frame', 'fy', 'start', 'val']

sample observations (period, days, fp, form, val):
  2006-12-02 -> 2007-11-30  363d   FY  10-K    3,157,881,000
  2007-11-30 -> 2008-05-30  182d   Q2  10-Q    1,777,331,000
  2008-02-29 -> 2008-05-30   91d   Q2  10-Q    886,886,000
  2007-12-01 -> 2008-08-29  272d   Q3  10-Q    2,664,588,000
  2008-05-31 -> 2008-08-29   90d   Q3  10-Q    887,257,000
  2007-12-01 -> 2008-11-28  363d   FY  10-K    3,579,889,000


## 4. Why `fy` cannot be trusted as the observation year

The raw `fy` field is the **filing's** fiscal year, not the economic period being reported. Each 10-K restates prior years as comparatives, so one economic period reappears in several filings with a different `fy` and accession but the same value.

Below, we group full-year observations by their economic period (`start`, `end`) and show how many distinct `fy` labels each one carries.

In [ ]:
from collections import defaultdict

by_period = defaultdict(list)
for fact in usd:
    if fact.get("fp") == "FY" and 350 <= duration_days(fact) <= 380:
        by_period[(fact["start"], fact["end"])].append(fact)

# Take one economic period that repeats across filings.
repeated = max(by_period.items(), key=lambda kv: len(kv[1]))
period, facts = repeated
print(f"economic period {period[0]} -> {period[1]}")
print(f"reported in {len(facts)} filings, all with the same value:\n")
for fact in sorted(facts, key=lambda f: f["filed"]):
    print(
        f"  filing fy={fact['fy']}  filed={fact['filed']}  "
        f"accn={fact['accn']}  val={fact['val']:,}"
    )

print("\nSame economic year, several `fy` labels. Trusting `fy` would duplicate the year.")
print("OwnerLens instead derives the fiscal year from the period end date.")

## 5. Run `normalize_annual_revenue()`

The normalizer selects one concept, keeps only full fiscal-year USD observations, derives each year from the period end date, collapses identical comparatives to the earliest original filing, and fails loudly on genuine conflicts.

In [ ]:
series = normalize_annual_revenue(raw)
print("ticker: ", series.ticker)
print("concept:", series.concept)
print("years:  ", [obs.fiscal_year for obs in series.observations])

## 6. Display canonical revenue + provenance

Each canonical observation carries the provenance needed to reconcile it back to a specific filing: concept, unit, economic period, form, filing date, and accession number.

In [ ]:
header = f"{'FY':<6}{'revenue (USD)':>18}   {'period':<26}{'form':<8}{'filed':<12}{'accession'}"
print(header)
print("-" * len(header))
for obs in series.observations:
    period = f"{obs.period_start} -> {obs.period_end}"
    print(
        f"{obs.fiscal_year:<6}{obs.value:>18,}   {period:<26}"
        f"{obs.form:<8}{str(obs.filed):<12}{obs.accession}"
    )

print("\nNote: each year resolves to its original 10-K (earliest filed), not a later comparative.")

## Takeaway

> A filing observation is not automatically a canonical financial fact. OwnerLens must identify the economic period, distinguish full-year observations, resolve duplicate comparative disclosures, and preserve provenance.